# Principal Component Analysis

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import IncrementalPCA
import joblib

## Dataset class

In [ ]:
class LatentSpaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data = []

        self._load_data()

    def _load_data(self):
        for filename in os.listdir(self.root_dir):
            _, ext = os.path.splitext(filename)
            if ext == '.pt':
                ls_path = os.path.join(self.root_dir, filename)
                self.data.append(ls_path)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ls_path = self.data[idx]
        latent_space = torch.load(ls_path)
        
        if self.transform:
            latent_space = self.transform(latent_space)

        return latent_space

In [64]:
dataset = LatentSpaceDataset(root_dir='latent-space-vectors')
print(len(dataset))
print(dataset[0])

100
tensor([0.4300, 0.8087, 0.6138,  ..., 0.1274, 0.7164, 0.5804])


## DataLoader class

In [65]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
latent_spaces = next(iter(dataloader))
print(latent_spaces.shape)

torch.Size([32, 8192])


In [66]:
print(latent_spaces[0])

tensor([0.4308, 0.0859, 0.1018,  ..., 0.2829, 0.5781, 0.4640])


## Feature extractor class

In [ ]:
class FeatureExtractor:
  def __init__(self, n_components=None):
    self.model = IncrementalPCA(n_components=n_components)

  def fit(self, dataset, batch_size=32):
    dataloader = DataLoader(dataset, batch_size=batch_size)
    for latent_spaces in dataloader:
      self.model.partial_fit(latent_spaces)

  def transform(self, latent_spaces):
    reduced_latent_spaces = self.model.transform(latent_spaces)
    return torch.tensor(reduced_latent_spaces)

  def save(self):
    filename = os.path.join('out', 'feature-extractor.pkl')
    joblib.dump(self, filename, 3)

  @staticmethod
  def load(filename):
    return joblib.load(filename)


In [72]:
fext = FeatureExtractor()
fext.fit(dataset)

In [75]:
print(dataset[0])
print(fext.transform(dataset[0].view(1, -1)))

tensor([0.4300, 0.8087, 0.6138,  ..., 0.1274, 0.7164, 0.5804])
tensor([[ 2.0665,  1.0814,  1.7259, -6.3500,  1.0909,  2.0796,  2.2409, -0.6911,
          0.1282,  2.0747,  2.7339,  3.9206, -2.0884,  0.7988, -2.9983,  0.8470,
         -2.2385, -2.0630,  1.4599, -1.4175, -4.5127, -5.5166, -4.2274, -1.4008,
         -4.4257,  1.4366, -0.4006,  2.1939, -3.4629,  2.5185, -1.0011, -2.4152]],
       dtype=torch.float64)


In [71]:
fext.save()